In [2]:
import pandas as pd
import numpy as np
import os

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_FILE = '30_gene_to_mz_synced_results_v1_analytic_fast/gene_to_mz_top30_matches_all_scores.csv' 
VALIDATION_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/2_mz_synced_isotope_80_matching_results/identified_isotopes.csv"
OUTPUT_FILE = '30_gene_to_mz_synced_results_v1_analytic_fast/final_gene_to_mz_top30_matches_all_scores.csv'

# RRF Constant (k): 
# k=0 makes Rank 1 much more valuable than Rank 2 (aggressive).
# k=60 makes Rank 1 and Rank 10 closer in value (smoother).
# For Top 10 lists, k=1 is usually a good balance.
K_FACTOR = 1
GROUPS = ['AAD', 'YC', 'AC', 'YAD'] 
TOLERANCE = 0.015  

DIFFS = {
    'Isotope (M+1)': 1.003355,   
    'Isotope (M+2)': 2.006710,   
    'Adduct (NH4)':  17.02655,   
    'Adduct (Na)':   21.98204,   
    'Adduct (K)':    37.95548,   
    'Loss (H2O)':    -18.01056   
}

# =============================================================================
# LOAD VALIDATION DATA (CHECKPOINT PREP)
# =============================================================================
def load_validation_lookup(path):
    try:
        val_df = pd.read_csv(path)
        # Create a set of frozen sets for order-independent lookup: {frozenset({mz1, mz2}), ...}
        # Rounding to 4 decimals to handle float precision differences
        lookup = set()
        for _, row in val_df.iterrows():
            pair = frozenset([round(row['mz_1'], 4), round(row['mz_2'], 4)])
            lookup.add(pair)
        print(f"Loaded {len(lookup)} validated isotope pairs.")
        return lookup
    except Exception as e:
        print(f"Error loading validation file: {e}")
        return set()

VALID_ISOTOPE_PAIRS = load_validation_lookup(VALIDATION_FILE)

# =============================================================================
# HELPER: CHECK RELATIONSHIPS
# =============================================================================
def get_relationship(parent_mz, child_mz):
    diff = child_mz - parent_mz
    potential_rel = None
    
    # 1. Determine Mathematical Relationship
    if abs(diff - DIFFS['Isotope (M+1)']) < TOLERANCE: potential_rel = "M+1"
    elif abs(diff - DIFFS['Isotope (M+2)']) < TOLERANCE: potential_rel = "M+2"
    elif abs(diff - DIFFS['Adduct (NH4)']) < TOLERANCE: potential_rel = "NH4"
    elif abs(diff - DIFFS['Adduct (Na)']) < TOLERANCE: potential_rel = "Na"
    elif abs(diff - DIFFS['Adduct (K)']) < TOLERANCE: potential_rel = "K"
    elif abs(diff - DIFFS['Loss (H2O)']) < TOLERANCE: potential_rel = "H2O Loss"
    
    # Sodium vs Ammonium check
    na_nh4_diff = DIFFS['Adduct (Na)'] - DIFFS['Adduct (NH4)']
    if potential_rel is None and abs(diff - na_nh4_diff) < TOLERANCE: 
        potential_rel = "Na (vs NH4)"
    
    # 2. CHECKPOINT: Only allow Isotope relationships if they exist in the validation file
    if potential_rel in ["M+1", "M+2"]:
        pair = frozenset([round(parent_mz, 4), round(child_mz, 4)])
        if pair not in VALID_ISOTOPE_PAIRS:
            return None # Reject mathematical match because it's not validated
            
    return potential_rel

def identify_group(sample_name):
    for g in GROUPS:
        if g in sample_name:
            return g
    return "Other"

# =============================================================================
# CONSENSUS ALGORITHM
# =============================================================================
def calculate_consensus(df, gene_name):
    subset = df[df['Gene'] == gene_name]
    if subset.empty: return None
    
    # Calculate Total Samples per Group (for denominator)
    all_samples = subset['Sample'].unique()
    total_samples_overall = len(all_samples)
    
    total_counts_per_group = {g: 0 for g in GROUPS + ['Other']}
    for s in all_samples:
        total_counts_per_group[identify_group(s)] += 1
    
    # 1. First Pass: Gather raw data per m/z
    mz_data = {} 
    for _, row in subset.iterrows():
        mz = row['MZ_Feature']
        sample = row['Sample']
        rank = row['Rank']
        group = identify_group(sample)
        
        if mz not in mz_data:
            mz_data[mz] = {
                'samples': set(), 
                'group_samples': {g: set() for g in GROUPS + ['Other']},
                'rrf': 0.0
            }
        mz_data[mz]['samples'].add(sample)
        mz_data[mz]['group_samples'][group].add(sample)
        mz_data[mz]['rrf'] += 1.0 / (rank + K_FACTOR)

    if not mz_data: return None

    # 2. Second Pass: Unify Counts (Parent absorbs Children)
    candidates = list(mz_data.keys())
    final_stats = []
    
    for parent in candidates:
        # Clone sets
        consolidated_all = mz_data[parent]['samples'].copy()
        consolidated_groups = {g: mz_data[parent]['group_samples'][g].copy() for g in GROUPS + ['Other']}
        children_found = []
        
        for child in candidates:
            if parent == child: continue
            rel = get_relationship(parent, child)
            if rel:
                # Merge ALL samples
                consolidated_all.update(mz_data[child]['samples'])

                # Merge GROUP samples
                for g in GROUPS + ['Other']:
                    consolidated_groups[g].update(mz_data[child]['group_samples'][g])
                children_found.append(f"{rel}")

        children_found = list(set(children_found))

        # Build Group Strings (e.g., "AAD:3/4")
        group_str_parts = []
        group_counts_numeric = {}
        for g in GROUPS:
            found = len(consolidated_groups[g])
            total = total_counts_per_group[g]
            group_str_parts.append(f"{g}:{found}/{total}")
            group_counts_numeric[g] = found

        final_stats.append({
            'mz': parent,
            'rrf': mz_data[parent]['rrf'],
            'total_count': len(consolidated_all),
            'group_counts': group_counts_numeric,
            'group_display': " | ".join(group_str_parts),
            'children': ", ".join(children_found)
        })

    # 3. Sort
    final_stats.sort(key=lambda x: x['rrf'], reverse=True)
    winner = final_stats[0]
    
    # Check if winner is potentially an artifact itself
    chem_flag = "Primary"
    for other in candidates:
        if other == winner['mz']: continue
        rel = get_relationship(other, winner['mz'])
        if rel:
            chem_flag = f"{rel} of {other:.4f}"
            break

    return {
        'winner': winner['mz'],
        'data': winner,
        'chem_flag': chem_flag,
        'total_samples_overall': total_samples_overall
    }

# =============================================================================
# MAIN EXECUTION
# =============================================================================
try:
    df = pd.read_csv(INPUT_FILE)
    df.columns = [c.title() for c in df.columns]
    if 'Rna_Sample' in df.columns: df.rename(columns={'Rna_Sample': 'Sample'}, inplace=True)
    if 'Mz_Feature' in df.columns: df.rename(columns={'Mz_Feature': 'MZ_Feature'}, inplace=True)
except FileNotFoundError:
    print(f"Error: Could not find '{INPUT_FILE}'.")
    exit()

unique_genes = df['Gene'].unique()
print(f"\n{'GENE':<15} | {'WINNER':<10} | {'TOTAL':<8} | {'CONF':<8} | {'GROUP BREAKDOWN':<45} | {'NOTES'}")
print("-" * 150)

results = []
for gene in unique_genes:
    res = calculate_consensus(df, gene)
    if not res: continue
    
    w = res['data']
    total_found = w['total_count']
    total_possible = res['total_samples_overall']
    
    # Calculate Confidence
    pct = (total_found / total_possible) * 100
    confidence = "LOW"
    if pct >= 50: confidence = "MEDIUM"
    if pct >= 75: confidence = "HIGH"
    if pct >= 90: confidence = "V.HIGH"
    
    total_str = f"{total_found}/{total_possible}"

    # Notes
    notes = []
    if res['chem_flag'] != "Primary": notes.append(f"⚠️ {res['chem_flag']}")
    if w['children']: notes.append(f"Incl: {w['children']}")

    print(f"{gene:<15} | {res['winner']:<10.4f} | {total_str:<8} | {confidence:<8} | {w['group_display']:<45} | {'; '.join(notes)}")
    
    # Save Dict
    row_data = {
        'Gene': gene,
        'Consensus_MZ': res['winner'],
        'Total_Found': total_found,
        'Total_Possible': total_possible,
        'Confidence': confidence,
        'Isotope_Flag': res['chem_flag'],
        'Merged_Isotopes': w['children']
    }
    # Add dynamic group columns
    for g in GROUPS:
        row_data[f'Count_{g}'] = w['group_counts'][g]
    results.append(row_data)

# Save
pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
print("-" * 150)
print(f"Complete. Results saved to {OUTPUT_FILE}")

Loaded 531 validated isotope pairs.

GENE            | WINNER     | TOTAL    | CONF     | GROUP BREAKDOWN                               | NOTES
------------------------------------------------------------------------------------------------------------------------------------------------------
AC149090.1      | 821.5308   | 9/16     | MEDIUM   | AAD:0/4 | YC:3/4 | AC:3/4 | YAD:3/4           | ⚠️ M+1 of 820.5250
Aplp1           | 628.5358   | 6/16     | LOW      | AAD:2/4 | YC:0/4 | AC:1/4 | YAD:3/4           | ⚠️ M+1 of 627.5340; Incl: M+1
Apoe            | 780.5501   | 6/16     | LOW      | AAD:0/4 | YC:4/4 | AC:1/4 | YAD:1/4           | ⚠️ Na of 758.5688; Incl: M+1
App             | 755.5406   | 10/16    | MEDIUM   | AAD:3/4 | YC:4/4 | AC:2/4 | YAD:1/4           | ⚠️ M+1 of 754.5364; Incl: M+2, M+1
Atp1a3          | 878.5697   | 9/16     | MEDIUM   | AAD:3/4 | YC:1/4 | AC:1/4 | YAD:4/4           | ⚠️ Na of 856.5819
Elavl3          | 858.6960   | 10/16    | MEDIUM   | AAD:4/4 | YC:0/4